In [ ]:
# 1. 라이브러리 설치
!pip install -q transformers accelerate scikit-learn peft trl bitsandbytes

import os
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import TrainingArguments
from google.colab import drive
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

# Seed for reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 45.2 MB/s eta 0:00:00
Device: cuda


In [ ]:
#cell 2

from huggingface_hub import login
login()

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.2-1B"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)
model.eval()

# Punctuation ID mapping
PUNCT_MAP = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("?", add_special_tokens=False)[-1]
}
EOS_ID = tokenizer.eos_token_id
print(f"Punctuation IDs: {PUNCT_MAP}")

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Punctuation IDs: {'COMMA': 11, 'PERIOD': 13, 'QMARK': 30}


In [ ]:
# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. User-provided path and loading code
BASE_PATH = "/content/drive/MyDrive/punct_data/iwslt2017/xy_label_v2"
VAL_Y_PATH = os.path.join(BASE_PATH, "iwslt2017_en_validation.Y.txt")
TEST_Y_PATH = os.path.join(BASE_PATH, "iwslt2017_en_test.Y.txt")
TRAIN_Y_PATH = os.path.join(BASE_PATH, "iwslt2017_en_train.Y.txt")

# Path to save fine-tuning results
OUTPUT_DIR = "/content/drive/MyDrive/experiments/llama_finetune0122"

print(f"Train Data Path: {TRAIN_Y_PATH}")
print(f"Output Model Path: {OUTPUT_DIR}")

print(f"Loading file: {VAL_Y_PATH}")
with open(VAL_Y_PATH, "r", encoding="utf-8") as f:
    val_y_list = [line.strip() for line in f if line.strip()]

with open(TEST_Y_PATH, "r", encoding="utf-8") as f:
    test_y_list = [line.strip() for line in f if line.strip()]


Mounted at /content/drive
Train Data Path: /content/drive/MyDrive/punct_data/iwslt2017/xy_label_v2/iwslt2017_en_train.Y.txt
Output Model Path: /content/drive/MyDrive/experiments/llama_finetune0122
Loading file: /content/drive/MyDrive/punct_data/iwslt2017/xy_label_v2/iwslt2017_en_validation.Y.txt


In [ ]:
# 3. Data loading function (Dataset definition issue fixed)
def load_full_dataset(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"can't find: {file_path}")

    with open(file_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    # Use Dataset imported above
    return Dataset.from_dict({"text": lines})

# 4. Execute
print(f"Loading data from: {TRAIN_Y_PATH}")
train_dataset = load_full_dataset(TRAIN_Y_PATH)
print(f"Dataset Loaded! Sample count: {len(train_dataset)}")

Loading data from: /content/drive/MyDrive/punct_data/iwslt2017/xy_label_v2/iwslt2017_en_train.Y.txt
Dataset Loaded! Sample count: 357117


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16 # Use FP16 for memory efficiency
)

In [ ]:
# 5. LoRA Configuration
# ---------------------------------------------------------
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"] # Tune core modules
)

In [ ]:
# 6. [Key Change] SFTConfig Configuration
# Use SFTConfig instead of TrainingArguments, and move related arguments here.
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",    # [Moved] Dataset column name
    # max_seq_length=256,           # [Moved] Sentence length limit
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=50,
    fp16=True,
    save_strategy="epoch",
    report_to="none"
    # packing=False, # (Use if needed, default False)
)

In [ ]:
# 7. Run Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=sft_config,              # Pass SFTConfig
    processing_class=tokenizer,   # [Modified] tokenizer -> processing_class
    peft_config=peft_config,
)

print(">>> Start Full Fine-tuning...")
trainer.train()

# 8. Save
trainer.save_model(OUTPUT_DIR)
print(f">>> Model Saved to {OUTPUT_DIR}")

# Clean up memory
del model, trainer
torch.cuda.empty_cache()

Exception ignored in: <function _xla_gc_callback at 0x79dee629d440>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/lib/__init__.py", line 127, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 


KeyboardInterrupt: 

In [ ]:
# ======================================================
# [Step 2] Load Fine-tuned Model (for Evaluation)
# ======================================================
from peft import PeftModel

# 1. Load Base Model again (skeleton)
# Load with FP16, same as during training.
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, # "meta-llama/Llama-3.2-1B"
    device_map="auto",
    torch_dtype=torch.float16
)

# 2. Combine trained LoRA Adapter (core)
# Overwrite the base model with the trained results saved in OUTPUT_DIR.
# The variable name should be 'model' for the existing experimental code to work.
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval() # Switch to evaluation mode (e.g., disable Dropout)

print(">>> Fine-tuned Llama-1B Loaded Successfully for Inference!")

# 3. Check tokenizer (if necessary)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

>>> Fine-tuned Llama-1B Loaded Successfully for Inference!


In [ ]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score
import re

from peft import PeftModel

# --- 1. Configuration ---
# Use full range to secure question mark (QMARK) samples
dataset_subset = val_y_list
CALIB_SIZE = len(dataset_subset)
K_VALUE = 2  # K=2 Tokens (Strict)

# Punctuation token IDs
PUNCT_TOKEN_IDS = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("?", add_special_tokens=False)[-1],
}
PUNCT_LIST = list(PUNCT_TOKEN_IDS.items())

def parse_sentence_to_boundaries(text: str):
    tokens = text.strip().split()
    boundaries = []
    for tok in tokens:
        # Pattern: (content)(punctuation)(zero or more closing quotes/parentheses/brackets)
        m = re.match(r'^(.*?)([,.?])(["\'\]\}]*)$', tok)

        if m:
            base, punct_char = m.group(1), m.group(2)

            if punct_char == ',':
                label = "COMMA"
            elif punct_char == '.':
                label = "PERIOD"
            elif punct_char == '?':
                label = "QMARK"

            word = base
        else:
            label = "O"
            word = tok

        # Remove quotes/parentheses from the word (minimum necessary)
        word = re.sub(r'[\"\'({\[\]}]', '', word)

        # Ignore tokens that are just quotes (e.g., ") as boundaries
        if word:
            boundaries.append({"word": word, "label": label})
            continue

        # Case 2: Punctuation-only tokens without a word (e.g., '."', ',"', '?")
        # This punctuation is assigned to the label of the preceding word.
        if m and boundaries:
            # If the previous label is already a punctuation, a policy is needed whether to overwrite or maintain.
            # Usually, the last punctuation is stronger, so overwriting is recommended.
            boundaries[-1]["label"] = label

        # Case 3: Ignore tokens that are just quotes (") etc.
        # (If m is not present and word is empty, it comes here)

    return boundaries

# Helper function: Calculate joint probability using chunking
def get_joint_score_optimized(prefix_ids, target_ids):
    """
    prefix_ids: Context tokens so far
    target_ids: Next K tokens (lookahead_tokens)
    """
    if not target_ids: return 0.0
    full_input = torch.tensor([prefix_ids + target_ids], device=device)
    with torch.no_grad():
        out = model(full_input)

    start_pos = len(prefix_ids) - 1
    # Extract Logits corresponding to the next K tokens
    rel_logits = out.logits[0, start_pos : start_pos + len(target_ids), :]
    log_probs = torch.log_softmax(rel_logits, dim=-1)
    t_ids_tensor = torch.tensor(target_ids, device=device)

    # Return sum of probabilities (Joint Probability) for those tokens
    return log_probs.gather(1, t_ids_tensor.unsqueeze(1)).squeeze(1).sum().item()

KeyboardInterrupt: 

In [ ]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score
import re

# --- 1. Configuration ---
# Use full range to secure question mark (QMARK) samples
dataset_subset = val_y_list
CALIB_SIZE = len(dataset_subset)
K_VALUE = 2  # K=2 Tokens (Strict)

# Punctuation token IDs
PUNCT_TOKEN_IDS = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("?", add_special_tokens=False)[-1],
}
PUNCT_LIST = list(PUNCT_TOKEN_IDS.items())

# --- 2. Data Collection (Strict Logic applied) ---
print(f"Step 1: Collecting scores for K={K_VALUE} (Strict K Tokens, Full Future)...")
raw_data = []

# Check EOS token (for Padding)
EOS_ID = tokenizer.eos_token_id

for sent_idx, y_true in enumerate(tqdm(dataset_subset)):
    # Text parsing
    boundaries = parse_sentence_to_boundaries(y_true)

    if not boundaries: continue

    current_words = []
    for i, item in enumerate(boundaries):
        word = item['word']
        gold_label = item['label']
        current_words.append(word)

        # History Encoding
        history_text = " ".join(current_words)
        h_ids = tokenizer.encode(history_text, add_special_tokens=True)

        # [Safety Check] Check if tokenizer appends EOS (Llama usually doesn't)
        if len(h_ids) > 0 and h_ids[-1] == EOS_ID:
             # If EOS is at the end, remove it (to prevent model from thinking the sentence ended)
            h_ids = h_ids[:-1]

        # Prepare Lookahead Tokens (Full Future & Strict Padding)
        # Get all future words and tokenize them
        full_future_words = [b['word'] for b in boundaries[i+1:]] # All future words
        full_future_str = " ".join(full_future_words)

        if not full_future_str:
            # If there's no future text, fill with EOS only
            lookahead_tokens = [EOS_ID] * K_VALUE
        else:
            # Encode the entire string
            full_future_ids = tokenizer.encode(" " + full_future_str, add_special_tokens=False)

            # Take the first K tokens
            lookahead_tokens = full_future_ids[:K_VALUE]

            # [Strict Padding] If less than K tokens, pad with EOS
            if len(lookahead_tokens) < K_VALUE:
                padding_len = K_VALUE - len(lookahead_tokens)
                lookahead_tokens = lookahead_tokens + ([EOS_ID] * padding_len)

        # S0: No Punctuation Score
        s0 = get_joint_score_optimized(h_ids, lookahead_tokens)

        # Cost (Current Word Logits)
        with torch.no_grad():
            h_out = model(torch.tensor([h_ids], device=device))
        base_logprobs = torch.log_softmax(h_out.logits[0, -1, :], dim=-1)

        # Batch Candidates (Batching + Chunking)
        batch_input_ids = [h_ids + [pid] + lookahead_tokens for _, pid in PUNCT_LIST]
        batch_tensor = torch.tensor(batch_input_ids, device=device)
        with torch.no_grad():
            batch_out = model(batch_tensor)

        entry = {"gold": gold_label}
        for b_idx, (pname, pid) in enumerate(PUNCT_LIST):
            entry[f"{pname}_cost"] = base_logprobs[pid].item()

            # Extract Gain
            start_pos = len(h_ids)
            rel_logits = batch_out.logits[b_idx, start_pos : start_pos + len(lookahead_tokens), :]
            lprobs = torch.log_softmax(rel_logits, dim=-1)
            t_ids_tensor = torch.tensor(lookahead_tokens, device=device)
            p_joint_score = lprobs.gather(1, t_ids_tensor.unsqueeze(1)).squeeze(1).sum().item()

            entry[f"{pname}_gain"] = p_joint_score - s0

        raw_data.append(entry)

        # Teacher Forcing Update
        if gold_label != "O":
            punct_char = "," if gold_label == "COMMA" else ("." if gold_label == "PERIOD" else "?")
            current_words[-1] = current_words[-1] + punct_char

df_scores = pd.DataFrame(raw_data)

# --- 3. Grid Search (Punctuation-Focused Metric) ---
print("\nStep 2: Searching for optimal parameters...")
alpha_range = np.arange(0.1, 0.95, 0.05)
threshold_range = np.arange(-3.0, 2.0, 0.25)

best_score = -1
best_params = {}
golds = df_scores["gold"].values

# Set evaluation labels excluding 'O'
EVAL_LABELS = ["COMMA", "PERIOD", "QMARK"]

for alpha in alpha_range:
    for thresh in threshold_range:
        preds = []
        for _, row in df_scores.iterrows():
            best_p, max_s = "O", float("-inf")
            for pname in ["COMMA", "PERIOD", "QMARK"]:
                # Use Convex Combination as mentioned in the paper (experimental tuning)
                score = (alpha * row[f"{pname}_cost"]) + ((1 - alpha) * row[f"{pname}_gain"])
                if score > max_s:
                    max_s, best_p = score, pname
            if max_s <= thresh: best_p = "O"
            preds.append(best_p)

        # Optimize punctuation performance using Macro F1 excluding 'O'
        # Set 'average="macro"' to give importance to the performance of minority classes (like QMARK)
        current_score = f1_score(golds, preds, average="macro", labels=EVAL_LABELS, zero_division=0)

        if current_score > best_score:
            best_score, best_params = current_score, {"alpha": alpha, "threshold": thresh}

print(f"\n=== K={K_VALUE} (Strict Token) Optimization Results ===")
print(f"Target Metric: Macro F1 (excluding 'O')")
print(f"Best Macro F1: {best_score:.4f}")
print(f"Optimal ALPHA: {best_params['alpha']:.2f}")
print(f"Optimal THRESHOLD: {best_params['threshold']:.2f}")

Step 1: Collecting scores for K=2 (Strict K Tokens, Full Future)...


100%|██████████| 1501/1501 [39:18<00:00,  1.57s/it]



Step 2: Searching for optimal parameters...

=== K=2 (Strict Token) Optimization Results ===
Target Metric: Macro F1 (excluding 'O')
Best Macro F1: 0.9094
Optimal ALPHA: 0.55
Optimal THRESHOLD: -0.25


In [ ]:
import torch
import pandas as pd
import time
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import re
import numpy as np

# --- 1. Parameter Settings (K=4 Optimal Values) ---
ALPHA_K4 = 0.55       # Calibration Result
THRESHOLD_K4 = -0.25  # Calibration Result
K_VALUE = 2           # Strict K=4

# Output order
LABELS_ORDER = ["O", "COMMA", "PERIOD", "QMARK"]

# Punctuation token IDs
PUNCT_TOKEN_IDS = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("?", add_special_tokens=False)[-1],
}
PUNCT_LIST = list(PUNCT_TOKEN_IDS.items())


# --- 2. Evaluation Loop (2-Pass Optimization applied) ---
print(f"Starting Final Evaluation (K={K_VALUE}, Strict Token Logic, 2-Pass Optimization)")
print(f"Params: Alpha={ALPHA_K4}, Threshold={THRESHOLD_K4}")
print("Metric: Words/second")

all_golds = []
all_preds = []
total_processed_words = 0
start_time = time.time()
EOS_ID = tokenizer.eos_token_id

for sent_idx, y_true in enumerate(tqdm(test_y_list)):
    boundaries = parse_sentence_to_boundaries(y_true)
    if not boundaries: continue

    current_words = []
    for i, item in enumerate(boundaries):
        word, gold_label = item['word'], item['label']
        current_words.append(word)
        total_processed_words += 1

        # History Encoding
        history_text = " ".join(current_words)
        h_ids = tokenizer.encode(history_text, add_special_tokens=True)
        if len(h_ids) > 0 and h_ids[-1] == EOS_ID: h_ids = h_ids[:-1]

        # [Strict K=4 Lookahead]
        full_future_words = [b['word'] for b in boundaries[i+1:]]
        full_future_str = " ".join(full_future_words)

        if not full_future_str:
            lookahead_tokens = [EOS_ID] * K_VALUE
        else:
            full_future_ids = tokenizer.encode(" " + full_future_str, add_special_tokens=False)
            lookahead_tokens = full_future_ids[:K_VALUE]
            if len(lookahead_tokens) < K_VALUE:
                lookahead_tokens = lookahead_tokens + ([EOS_ID] * (K_VALUE - len(lookahead_tokens)))

        # === Core of 2-Pass Optimization ===
        # Pass 1: Combine History + Lookahead to get (Cost) and (S0) simultaneously
        # -------------------------------------------------------------------------
        # Input structure: [h_1, ..., h_n, t_1, ..., t_k]
        full_input_ids = h_ids + lookahead_tokens
        full_input_tensor = torch.tensor([full_input_ids], device=device)

        with torch.no_grad():
            out_1 = model(full_input_tensor)

        # (1) Extract Cost: Logits at the position of the last token of History (h_n)
        # Position of h_n is index `len(h_ids) - 1`
        # These Logits represent "next token prediction" probability, from which punctuation probabilities (Cost) are obtained.
        base_logits = out_1.logits[0, len(h_ids) - 1, :]
        base_logprobs = torch.log_softmax(base_logits, dim=-1)

        # (2) Extract S0: Probability from the end of History to the end of Lookahead (Joint Probability)
        # Logits range: `len(h_ids) - 1` (first Lookahead prediction) ~ `len(full_input) - 2` (last Lookahead prediction)
        start_pos = len(h_ids) - 1
        target_len = len(lookahead_tokens)

        # Extract Logits for the Lookahead part
        # out_1.logits[0, start_pos : start_pos + target_len] -> [K, Vocab]
        s0_logits = out_1.logits[0, start_pos : start_pos + target_len, :]
        s0_logprobs = torch.log_softmax(s0_logits, dim=-1)
        s0_target_ids = torch.tensor(lookahead_tokens, device=device)

        # Sum probabilities of correct tokens (Lookahead tokens)
        s0 = s0_logprobs.gather(1, s0_target_ids.unsqueeze(1)).squeeze(1).sum().item()

        # Pass 2: Batch Forward (Calculate Gain)
        # -------------------------------------------------------------------------
        batch_input_ids = [h_ids + [pid] + lookahead_tokens for _, pid in PUNCT_LIST]
        batch_tensor = torch.tensor(batch_input_ids, device=device)

        with torch.no_grad():
            batch_out = model(batch_tensor)

        best_p, max_s = "O", float('-inf')
        for b_idx, (pname, pid) in enumerate(PUNCT_LIST):
            # Use Cost calculated in Pass 1
            cost = base_logprobs[pid].item()

            # Extract Gain
            # Batch input structure: [h_1...h_n, PUNCT, t_1...t_k]
            # PUNCT position is len(h_ids). t_1 prediction comes from Logit at len(h_ids) position.
            start_pos_batch = len(h_ids)
            rel_logits = batch_out.logits[b_idx, start_pos_batch : start_pos_batch + len(lookahead_tokens), :]
            lprobs = torch.log_softmax(rel_logits, dim=-1)
            t_ids_tensor = torch.tensor(lookahead_tokens, device=device)
            p_joint_score = lprobs.gather(1, t_ids_tensor.unsqueeze(1)).squeeze(1).sum().item()

            gain = p_joint_score - s0

            score = (ALPHA_K4 * cost) + ((1 - ALPHA_K4) * gain)

            if score > THRESHOLD_K4 and score > max_s:
                max_s, best_p = score, pname

        all_golds.append(gold_label)
        all_preds.append(best_p)

        if best_p != "O":
            punct_char = "," if best_p == "COMMA" else ("." if best_p == "PERIOD" else "?")
            current_words[-1] = current_words[-1] + punct_char

# --- 3. Result Report ---
end_time = time.time()
elapsed = end_time - start_time
wps = total_processed_words / elapsed

print(f"\n[Final Results | K={K_VALUE} Strict 2-Pass Optimized]")
print(f"Inference Speed: {wps:.2f} words/s")
print(f"Total Processed Words: {total_processed_words}")
print(f"Total Execution Time: {elapsed:.2f}s")
print("-" * 60)
print(classification_report(all_golds, all_preds, labels=LABELS_ORDER, zero_division=0, digits=3))

print("\nConfusion Matrix")
cm = confusion_matrix(all_golds, all_preds, labels=LABELS_ORDER)
df_cm = pd.DataFrame(cm, index=[f"True_{l}" for l in LABELS_ORDER], columns=[f"Pred_{l}" for l in LABELS_ORDER])
print(df_cm)

Starting Final Evaluation (K=2, Strict Token Logic, 2-Pass Optimization)
Params: Alpha=0.55, Threshold=-0.25
Metric: Words/second


100%|██████████| 10799/10799 [2:40:23<00:00,  1.12it/s]



[Final Results | K=2 Strict 2-Pass Optimized]
Inference Speed: 19.15 words/s
Total Processed Words: 184278
Total Execution Time: 9623.36s
------------------------------------------------------------
              precision    recall  f1-score   support

           O      0.987     0.986     0.987    160196
       COMMA      0.836     0.844     0.840     13017
      PERIOD      0.989     0.983     0.986     10153
       QMARK      0.949     0.922     0.935       912

    accuracy                          0.976    184278
   macro avg      0.940     0.934     0.937    184278
weighted avg      0.976     0.976     0.976    184278


Confusion Matrix
             Pred_O  Pred_COMMA  Pred_PERIOD  Pred_QMARK
True_O       158024        2145           23           4
True_COMMA     2010       10989           18           0
True_PERIOD     116          15         9981          41
True_QMARK        1           2           68         841
